## Demostración de Funciones 


 **Se demostrará el uso de todas las funciones**

### 1.  Configuración



In [180]:
import pandas as pd
import numpy as np
from typing import Iterable, List, Dict, Any

In [181]:
df = pd.read_csv("dataset/CTG.csv")

print(" DF original creado:")
print(df)

print("\nConteo inicial de Nulos:")
df.isnull().sum()

 DF original creado:
          FileName       Date      SegFile        b        e    LBE     LB  \
0     Variab10.txt  12/1/1996  CTG0001.txt   240.00   357.00 120.00 120.00   
1       Fmcs_1.txt   5/3/1996  CTG0002.txt     5.00   632.00 132.00 132.00   
2       Fmcs_1.txt   5/3/1996  CTG0003.txt   177.00   779.00 133.00 133.00   
3       Fmcs_1.txt   5/3/1996  CTG0004.txt   411.00 1,192.00 134.00 134.00   
4       Fmcs_1.txt   5/3/1996  CTG0005.txt   533.00 1,147.00 132.00 132.00   
...            ...        ...          ...      ...      ...    ...    ...   
2124  S8001045.dsp   6/6/1998  CTG2127.txt 1,576.00 3,049.00 140.00 140.00   
2125  S8001045.dsp   6/6/1998  CTG2128.txt 2,796.00 3,415.00 142.00 142.00   
2126           NaN        NaN          NaN      NaN      NaN    NaN    NaN   
2127           NaN        NaN          NaN      NaN      NaN    NaN    NaN   
2128           NaN        NaN          NaN      NaN      NaN    NaN    NaN   

       AC     FM    UC  ...    C    D    E

FileName    3
Date        3
SegFile     3
b           3
e           3
LBE         3
LB          3
AC          3
FM          2
UC          2
ASTV        2
MSTV        2
ALTV        2
MLTV        2
DL          1
DS          1
DP          1
DR          1
Width       3
Min         3
Max         3
Nmax        3
Nzeros      3
Mode        3
Mean        3
Median      3
Variance    3
Tendency    3
A           3
B           3
C           3
D           3
E           3
AD          3
DE          3
LD          3
FS          3
SUSP        3
CLASS       3
NSP         3
dtype: int64

 
## Definición de Funciones 


#### Función `eliminar_columnas_con_nulos()`
Elimina columnas enteras si la proporción de valores nulos excede un umbral (`umbral_porcentaje`), generalmente 20%. Esto ayuda a descartar características con demasiados datos faltantes.

In [182]:
def eliminar_columnas_con_nulos(df: pd.DataFrame, umbral_porcentaje: float = 0.20) -> pd.DataFrame:
    """Elimina columnas que superen un umbral de 20%  valores nulos."""
    if not 0 <= umbral_porcentaje <= 1:
        raise ValueError("umbral_porcentaje debe estar entre 0 y 1")
    num_filas = len(df)
    umbral = int(np.floor(umbral_porcentaje * num_filas))
    total_nulos = df.isnull().sum()
    col_eliminar = total_nulos[total_nulos > umbral].index.tolist()
    df_resultado = df.drop(columns=col_eliminar) if col_eliminar else df.copy()
    print(f"Columnas eliminadas (>{umbral_porcentaje*100:.1f}% nulos = {umbral} nulos): {len(col_eliminar)} columnas")
    return df_resultado

In [183]:
print(df.shape)
print("\nFunción eliminar_columnas_con_nulos definida.")
df_actual = eliminar_columnas_con_nulos(df.copy(), umbral_porcentaje=0.20)
print("\nDimensiones después de eliminar columnas con demasiados nulos:")
print(df_actual.shape)

(2129, 40)

Función eliminar_columnas_con_nulos definida.
Columnas eliminadas (>20.0% nulos = 425 nulos): 0 columnas

Dimensiones después de eliminar columnas con demasiados nulos:
(2129, 40)


La función eliminar_columnas_con_nulos evalúa cada columna y la elimina si la proporción de valores nulos excede el 20% del total de filas. Este paso preliminar descarta características de baja calidad. El output muestra cuántas columnas fueron eliminadas y las nuevas dimensiones del DataFrame (df_actual), confirmando que el proceso se realizó.

*Para este dataset, no se elimina ninguna columna. Todas las columnas importantes (LBE, AC, ASTV, etc.) tienen un 100% de completitud. Solo las columnas iniciales de texto (FileName, Date, etc.) tienen nulos, pero no exceden el umbral del 20% necesario para la eliminación.*

#### Función `imputar_valores()`
Rellena los valores nulos (`NaN`) restantes en las columnas. Utiliza la **mediana** o la **media** para las columnas numéricas y la **moda** para las categóricas.

In [ ]:
def imputar_valores(df: pd.DataFrame,
                    col_num: Iterable[str] = None,
                    col_cat: Iterable[str] = None,
                    metodo_num: str = "mediana") -> pd.DataFrame:
    """Imputa valores faltantes: numéricos: mediana o media, categóricos: moda."""
    
    df_res = df.copy()

    if col_num is None:
        col_num = df_res.select_dtypes(include=np.number).columns.tolist()
    if col_cat is None:
        col_cat = df_res.select_dtypes(include=['object', 'category']).columns.tolist()

    for col in col_num:
        cambios_col = df_res[col].isnull().sum()
        if cambios_col > 0:
            if metodo_num == "mediana":
                valor = df_res[col].median()
            elif metodo_num == "media":
                valor = df_res[col].mean()
            else:
                valor = df_res[col].median()
                
            df_res[col] = df_res[col].fillna(valor)
            print(f" Columna '{col}' (Numérica): {cambios_col} valores imputados, valor ({valor:.2f}). ")
    

    for col in col_cat:
        cambios_col = df_res[col].isnull().sum()
        if cambios_col > 0:
            moda = df_res[col].mode()
            if moda.empty:
                valor = ""
            else:
                valor = moda[0]
            df_res[col] = df_res[col].fillna(valor)
            print(f" Columna '{col}' (Categórica): {cambios_col} valores imputado, valor ({valor}).")

    return df_res

In [200]:
# Verificación de nulos restantes
print("Nulos por columna antes de Imputación ( > 0):")
nulos_antes_imputacion = df_actual.isnull().sum()
print(nulos_antes_imputacion[nulos_antes_imputacion > 0].sort_values(ascending=False))

df_imputado = imputar_valores(df=df_actual, metodo_num='mediana') # Aplicamos mediana a todas las numéricas por defecto 

print("\nNulos restantes después de la Imputación:")
nulos_despues_imputacion = df_imputado.isnull().sum()
display(nulos_despues_imputacion[nulos_despues_imputacion > 0].sort_values(ascending=False))

Nulos por columna antes de Imputación ( > 0):
FileName    3
Date        3
Nzeros      3
Mode        3
Mean        3
Median      3
Variance    3
Tendency    3
A           3
B           3
C           3
D           3
E           3
AD          3
DE          3
LD          3
FS          3
SUSP        3
CLASS       3
Nmax        3
Max         3
Min         3
Width       3
SegFile     3
b           3
e           3
LBE         3
LB          3
AC          3
NSP         3
MSTV        2
UC          2
FM          2
ALTV        2
MLTV        2
ASTV        2
DL          1
DS          1
DP          1
DR          1
dtype: int64
 Columna 'b' (Numérica): 3 valores imputados, valor (538.00). 
 Columna 'e' (Numérica): 3 valores imputados, valor (1241.00). 
 Columna 'LBE' (Numérica): 3 valores imputados, valor (133.00). 
 Columna 'LB' (Numérica): 3 valores imputados, valor (133.00). 
 Columna 'AC' (Numérica): 3 valores imputados, valor (1.00). 
 Columna 'FM' (Numérica): 2 valores imputados, valor (0.00). 
 

Series([], dtype: int64)

La función imputar_valores rellena los valores nulos (NaN) que no fueron eliminados en el paso anterior. Se utilizó la mediana para las columnas numéricas para mitigar el impacto de posibles outliers y la moda para las columnas categóricas, asegurando que todos los campos relevantes contengan datos. El output detalla los cambios por columna, y la verificación final de nulos demuestra la completitud.

*Rellena los nulos en las primeras tres columnas de texto (FileName, Date, SegFile) y en las columnas numéricas que tienen nulos (b, e, LBE, etc.). Por ejemplo, los nulos en la columna LBE se rellenan con su mediana.*

#### Función `verificar_unicos_outliers()`
Revisa las columnas numéricas para ver si tienen más de 10 valores únicos. Las que cumplen se marcan como APTAS para el análisis de outliers.

In [186]:
def verificar_unicos_outliers(df: pd.DataFrame) -> List[str]:
    """
    Verifica el número de valores únicos en las columnas numéricas.
    Devuelve la lista de columnas aptas (>= 10 valores únicos) para el análisis de outliers tradicional.
    """
    col_num = df.select_dtypes(include=np.number).columns

    col_aptas = []
    col_no_aptas = []

    for col in col_num:
        unique_count = df[col].nunique()

        if unique_count <= 10:
            print(f"**NO APTA** - {col}: {unique_count} valores únicos (Posiblemente variable categórica/ordinal)")
            col_no_aptas.append(col)
        else:
            print(f"**APTA** - {col}: {unique_count} valores únicos (Adecuada para análisis de outliers)")
            col_aptas.append(col)
   
    print(f"Columnas Aptas para Outliers ({len(col_aptas)}): {col_aptas}")
    print(f"Columnas No Aptas ({len(col_no_aptas)}): {col_no_aptas}")

    return col_aptas

In [187]:
print(" Función verificar_unicos_outliers")
cols_aptas_outliers = verificar_unicos_outliers(df_imputado)

 Función verificar_unicos_outliers
**APTA** - b: 979 valores únicos (Adecuada para análisis de outliers)
**APTA** - e: 1064 valores únicos (Adecuada para análisis de outliers)
**APTA** - LBE: 48 valores únicos (Adecuada para análisis de outliers)
**APTA** - LB: 48 valores únicos (Adecuada para análisis de outliers)
**APTA** - AC: 22 valores únicos (Adecuada para análisis de outliers)
**APTA** - FM: 96 valores únicos (Adecuada para análisis de outliers)
**APTA** - UC: 19 valores únicos (Adecuada para análisis de outliers)
**APTA** - ASTV: 75 valores únicos (Adecuada para análisis de outliers)
**APTA** - MSTV: 57 valores únicos (Adecuada para análisis de outliers)
**APTA** - ALTV: 87 valores únicos (Adecuada para análisis de outliers)
**APTA** - MLTV: 249 valores únicos (Adecuada para análisis de outliers)
**APTA** - DL: 15 valores únicos (Adecuada para análisis de outliers)
**NO APTA** - DS: 2 valores únicos (Posiblemente variable categórica/ordinal)
**NO APTA** - DP: 5 valores únicos (

*La mayoría de las métricas clave del CTG (ej. LBE, ASTV, MSTV, Max, Min, Width, Mode, Mean, Variance) se clasifican como APTAS (Continuas). Columnas como AC, FM, UC, DL, DS, DP se marcan como NO APTAS (Discretas) porque tienen muy pocos valores únicos (ej. conteos pequeños o valores binarios 0/1).*

#### Función `detectar_outliers_iqr()`
Calcula los límites superior e inferior utilizando el **Rango Intercuartílico (IQR)** (Q3 - Q1) multiplicado por 1.5. Luego, cuenta cuántos valores caen fuera de estos límites, reportando el número y el porcentaje de *outliers*.

In [188]:
def detectar_outliers_iqr(df: pd.DataFrame, columnas: Iterable[str] = None) -> Dict[str, Dict[str, float]]:
    """Detecta outliers por IQR para columnas numéricas seleccionadas."""
    
    if columnas is None:
        columnas = df.select_dtypes(include=np.number).columns.tolist()
    columnas_a_procesar = [col for col in columnas if col in df.columns and pd.api.types.is_numeric_dtype(df[col])]

    resultado = {}
    n = len(df)
    
    if n == 0:
        return resultado
        
    for col in columnas_a_procesar:
        serie = df[col].dropna() 
            
        Q1 = serie.quantile(0.25)
        Q3 = serie.quantile(0.75)
        IQR = Q3 - Q1
        
        if IQR == 0:
            continue
            
        lim_inf = Q1 - 1.5 * IQR
        lim_sup = Q3 + 1.5 * IQR
        
        num_outliers = int(((serie < lim_inf) | (serie > lim_sup)).sum())
        porcentaje_outliers =  (num_outliers / n) * 100
        
        resultado[col] = {
            "num_outliers": int(num_outliers),
            "porcentaje_outliers": float(porcentaje_outliers),
            "lim_inferior": float(lim_inf),
            "lim_superior": float(lim_sup)
        }

    
    return resultado

In [189]:
print("Función detectar_outliers_iqr ")
df_outliers_iqr = detectar_outliers_iqr(df_imputado, cols_aptas_outliers) 
print(df_outliers_iqr )

Función detectar_outliers_iqr 
{'b': {'num_outliers': 0, 'porcentaje_outliers': 0.0, 'lim_inferior': -2139.5, 'lim_superior': 3712.5}, 'e': {'num_outliers': 0, 'porcentaje_outliers': 0.0, 'lim_inferior': -1128.5, 'lim_superior': 4571.5}, 'LBE': {'num_outliers': 0, 'porcentaje_outliers': 0.0, 'lim_inferior': 105.0, 'lim_superior': 161.0}, 'LB': {'num_outliers': 0, 'porcentaje_outliers': 0.0, 'lim_inferior': 105.0, 'lim_superior': 161.0}, 'AC': {'num_outliers': 83, 'porcentaje_outliers': 3.8985439173320806, 'lim_inferior': -6.0, 'lim_superior': 10.0}, 'FM': {'num_outliers': 311, 'porcentaje_outliers': 14.607797087834665, 'lim_inferior': -3.0, 'lim_superior': 5.0}, 'UC': {'num_outliers': 23, 'porcentaje_outliers': 1.0803193987787694, 'lim_inferior': -5.0, 'lim_superior': 11.0}, 'ASTV': {'num_outliers': 0, 'porcentaje_outliers': 0.0, 'lim_inferior': -11.5, 'lim_superior': 104.5}, 'MSTV': {'num_outliers': 71, 'porcentaje_outliers': 3.3348990136214187, 'lim_inferior': -0.8, 'lim_superior': 3

Muestra un alto número de outliers en varias columnas.

#### Función `capping ()`
Corrige los *outliers* identificados. Los valores que exceden el límite superior se reemplazan con el valor del límite superior, y los que están por debajo del límite inferior se reemplazan con ese límite inferior. Esto **limita el impacto** de los valores extremos.

In [190]:
def capping(df: pd.DataFrame, columnas: Iterable[str] = None) -> pd.DataFrame:
    """
    Aplica el 'capping' (limitación) a los valores atípicos (outliers) 
    en las columnas numéricas especificadas usando el método 1.5 * IQR.
    """
    
    df_capping = df.copy()

  
    
    if columnas is None:
        cols_to_process: List[str] = df_capping.select_dtypes(include=np.number).columns.tolist()
    else:
        cols_to_process = list(columnas) 

    for col in cols_to_process:
        if col not in df_capping.columns or not np.issubdtype(df_capping[col].dtype, np.number):
            if col in df_capping.columns:
                print(f"Columna '{col}' no es numérica. Omitiendo.")
            continue
            
        Q1 = df_capping[col].quantile(0.25)
        Q3 = df_capping[col].quantile(0.75)
        IQR = Q3 - Q1
        
        lim_inferior = Q1 - 1.5 * IQR
        lim_superior = Q3 + 1.5 * IQR
        
        outliers_bajos = (df_capping[col] < lim_inferior).sum()
        outliers_altos = (df_capping[col] > lim_superior).sum()
        
        if outliers_bajos > 0 or outliers_altos > 0:
            df_capping[col] = df_capping[col].clip(lower=lim_inferior, upper=lim_superior)
            print(f"Columna '{col}': {outliers_bajos} bajos, valor {lim_inferior:.2f} | {outliers_altos} altos, valor {lim_superior:.2f}")

    return df_capping

In [191]:
print("Función capping ")
df_capping = capping(df_imputado, columnas=cols_aptas_outliers)
# Volvermos a verificar los outliers en las columnas procesadas (deben ser 0 o muy cercanos)
outlier_check = detectar_outliers_iqr(df_imputado, columnas=cols_aptas_outliers)

for col, info in outlier_check.items():
    print(f"Columna '{col}': {info['num_outliers']} outliers.")

Función capping 
Columna 'AC': 0 bajos, valor -6.00 | 83 altos, valor 10.00
Columna 'FM': 0 bajos, valor -3.00 | 311 altos, valor 5.00
Columna 'UC': 0 bajos, valor -5.00 | 23 altos, valor 11.00
Columna 'MSTV': 0 bajos, valor -0.80 | 71 altos, valor 3.20
Columna 'ALTV': 0 bajos, valor -16.50 | 310 altos, valor 27.50
Columna 'MLTV': 0 bajos, valor -4.70 | 72 altos, valor 20.10
Columna 'DL': 0 bajos, valor -4.50 | 82 altos, valor 7.50
Columna 'Max': 0 bajos, valor 119.00 | 24 altos, valor 207.00
Columna 'Nmax': 0 bajos, valor -4.00 | 19 altos, valor 12.00
Columna 'Mode': 61 bajos, valor 100.50 | 12 altos, valor 176.50
Columna 'Mean': 42 bajos, valor 95.00 | 3 altos, valor 175.00
Columna 'Median': 23 bajos, valor 100.50 | 5 altos, valor 176.50
Columna 'Variance': 0 bajos, valor -31.00 | 184 altos, valor 57.00
Columna 'b': 0 outliers.
Columna 'e': 0 outliers.
Columna 'LBE': 0 outliers.
Columna 'LB': 0 outliers.
Columna 'AC': 83 outliers.
Columna 'FM': 311 outliers.
Columna 'UC': 23 outliers

*El capping modifica  valores en columnas con alta dispersión como ALTV y MLTV, forzando a los valores más altos a ser iguales al límite superior, lo que reduce su impacto en el modelo*

#### Función `check_data_completeness_AldoAlbertoRodriguezFlores`
Genera el reporte final de calidad de datos. **Clasifica** cada columna en **Continua**, **Discreta** o **Categórica** (basado en el número de valores únicos y el tipo de dato) y calcula todos los estadísticos de calidad (nulos, completitud, IQR y *outliers*).

In [192]:
def check_data_completeness_AldoAlbertoRodriguezFlores(df: pd.DataFrame) -> pd.DataFrame:
    """Genera un DataFrame de resumen de calidad de datos y Clasificación."""
    
    resumen_dic: Dict[str, Dict[str, Any]] = {}
    
    for col in df.columns:
        num_nulos = df[col].isnull().sum()
        num_filas = len(df)
        porcen_nulos = (num_nulos / num_filas) * 100
        porcen_completo = 100 - porcen_nulos
        dtype_str = str(df[col].dtype)
        num_unicos = df[col].nunique()
        
        # Clasificación (misma lógica que verificar_unicos_outliers)
        if num_unicos > 10 and pd.api.types.is_numeric_dtype(df[col]):
            clasificacion = "Continua"
        elif num_unicos <= 10:
            clasificacion = "Discreta"
        else:
            clasificacion = "Categórica" 
            
        resumen_dic[col] = {
            "Conteo Nulos": int(num_nulos),
            "Porc de Completitud": float(porcen_completo),
            "Tipo de Dato": dtype_str,
            "Clasificación": clasificacion, 
            "Valores Únicos": int(num_unicos), 
            
            # Inicializacion 
            "IQR": np.nan,
            "Lim Inf Outlier": np.nan,
            "Lim Sup Outlier": np.nan,
            "Conteo Outliers": np.nan,
            "Porctentaje Outliers": np.nan,
        }

    df_resumen = pd.DataFrame.from_dict(resumen_dic, orient='index')
    
    # Dispersión (IQR y Outliers) - Solo para columnas Continuas
    col_continuas = df_resumen[df_resumen["Clasificación"] == "Continua"].index.tolist()
    
    if col_continuas:
        # Se usa la función de preprocessing.py definida anteriormente
        outlier_datos = detectar_outliers_iqr(df, columnas=col_continuas)

        for col, datos in outlier_datos.items():
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            
            df_resumen.loc[col, "IQR"] = float(IQR)
            df_resumen.loc[col, "Lim Inf Outlier"] = datos["lim_inferior"]
            df_resumen.loc[col, "Lim Sup Outlier"] = datos["lim_superior"]
            df_resumen.loc[col, "Conteo Outliers"] = datos["num_outliers"]
            df_resumen.loc[col, "Porctentaje Outliers"] = datos["porcentaje_outliers"]

    
    # Eliminar la columna auxiliar y ordenar las columnas para una mejor presentación
    df_resumen = df_resumen.drop(columns=["Valores Únicos"])
    
    col_orden = [
        "Conteo Nulos", "Porc de Completitud", "Tipo de Dato", "Clasificación",
        "IQR", "Lim Inf Outlier", "Lim Sup Outlier", "Conteo Outliers", 
        "Porctentaje Outliers"
    ]
    df_resumen = df_resumen[col_orden]
    
    pd.options.display.float_format = '{:,.2f}'.format
    
    return df_resumen.sort_values(by="Porctentaje Outliers", ascending=True)

In [193]:
df_resumen = check_data_completeness_AldoAlbertoRodriguezFlores(df_capping)

print("\n check_data_completeness_AldoAlbertoRodriguezFlores:")
display(df_resumen)


 check_data_completeness_AldoAlbertoRodriguezFlores:


,Conteo Nulos,Porc de Completitud,Tipo de Dato,Clasificación,IQR,Lim Inf Outlier,Lim Sup Outlier,Conteo Outliers,Porctentaje Outliers
b,0,100.00,float64,Continua,"1,463.00","-2,139.50","3,712.50",0.00,0.00
Mean,0,100.00,float64,Continua,20.00,95.00,175.00,0.00,0.00
Mode,0,100.00,float64,Continua,19.00,100.50,176.50,0.00,0.00
Nmax,0,100.00,float64,Continua,4.00,-4.00,12.00,0.00,0.00
Max,0,100.00,float64,Continua,22.00,119.00,207.00,0.00,0.00
Min,0,100.00,float64,Continua,53.00,-12.50,199.50,0.00,0.00
Width,0,100.00,float64,Continua,63.00,-57.50,194.50,0.00,0.00
MLTV,0,100.00,float64,Continua,6.20,-4.70,20.10,0.00,0.00
ALTV,0,100.00,float64,Continua,11.00,-16.50,27.50,0.00,0.00
MSTV,0,100.00,float64,Continua,1.00,-0.80,3.20,0.00,0.00


*El flujo de preprocesamiento revela que el conjunto de datos de monitoreo fetal casi no tiene valores nulos, por lo que las funciones de eliminar_columnas_con_nulos e imputar_valores confirman un 100% de integridad en las 40 variables. Sin embargo, la auditoría muestra que el principal problema de calidad residía en la dispersión extrema de las variables Continuas, como la variabilidad de largo y corto plazo (ALTV y MLTV).*

 *los valores atípicos se limitaron a los umbrales normales, dejando un dataset final que no solo está 100% completo, sino que también está estadísticamente más robusto contra errores de medición. La clasificación de check_data_completeness_AldoAlbertoRodriguezFlores separa las mediciones Continuas (listas para el análisis) de los contadores de eventos Discretos (que se tratarán como categorías).*

#### Función `columnas_numericas`
Esta función revisa el tipo de dato de cada columna en el DataFrame y devuelve una lista con los nombres de aquellas que son de naturaleza numérica (enteros o flotantes).

In [194]:
def columnas_numericas(df: pd.DataFrame) -> List[str]:
    """Retorna lista de columnas numéricas."""
    return df.select_dtypes(include=np.number).columns.tolist()

In [195]:
cols_num = columnas_numericas(df_imputado)
print(f"Columnas Numéricas: {cols_num}")


Columnas Numéricas: ['b', 'e', 'LBE', 'LB', 'AC', 'FM', 'UC', 'ASTV', 'MSTV', 'ALTV', 'MLTV', 'DL', 'DS', 'DP', 'DR', 'Width', 'Min', 'Max', 'Nmax', 'Nzeros', 'Mode', 'Mean', 'Median', 'Variance', 'Tendency', 'A', 'B', 'C', 'D', 'E', 'AD', 'DE', 'LD', 'FS', 'SUSP', 'CLASS', 'NSP']


#### Función `columnas_categoricas`
Esta función identifica y lista aquellas columnas cuyo tipo de dato es `object` o `category`, las cuales suelen representar variables categóricas o de texto.

In [198]:
def columnas_categoricas(df: pd.DataFrame) -> List[str]:
    """Retorna lista de columnas categóricas."""
    return df.select_dtypes(include=['object', 'category']).columns.tolist()

In [197]:
cols_cat = columnas_categoricas(df_imputado)
print(f"Columnas Categóricas/Object: {cols_cat}")

Columnas Categóricas/Object: ['FileName', 'Date', 'SegFile']
